## Deep Agents
`create_agent` gives you a tool-calling loop. For long, multi-step tasks (research, writing a report, coding) an agent also needs a place to keep notes, a way to delegate sub-tasks and protection against an overflowing context. **Deep Agents** (`deepagents`) is a ready-made agent harness built on LangChain and LangGraph that adds exactly that:
- **virtual file system** – tools `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`; files live in the agent state,
- **subagents** – the `task` tool delegates work to specialised agents with their own prompt and tools, so the main context stays small,
- **context management** – long conversations are summarised automatically (`SummarizationMiddleware`).

The result of `create_deep_agent` is a regular compiled LangGraph graph, so `invoke`, `stream`, checkpointers and LangSmith work as usual.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph/blob/main/notebooks/en/13_1_Deep_agents_en.ipynb)

| | |
|---|---|
| 🎯 **What you'll learn** | Deep Agents: an agent harness with a virtual file system, subagents and context management, built on LangGraph. |
| 🔑 **API keys** | `OPENAI_API_KEY`, `TAVILY_API_KEY` |
| 💲 **API cost** | low – a dozen or so requests |
| ⏱️ **Run time** | ~2 min |

### Install libraries

In [1]:
# Install this notebook's direct dependencies.
# Skipped when pip is missing, e.g. in the `uv sync` environment, which already has them (see README).
import importlib.util
if importlib.util.find_spec("pip"):
    %pip install -qU deepagents langchain-openai langchain-tavily python-dotenv
# For Tavily search, also add TAVILY_API_KEY to Colab Secrets or your local .env file.

Note: you may need to restart the kernel to use updated packages.


### Load env variables

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 1. The simplest deep agent and its file system
Even without our own tools the agent has the built-in file tools. We ask it to save the result to a file and then read the file from the returned state.

In [3]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="openai:gpt-5-mini",
    system_prompt="You are a helpful assistant. Save every deliverable as a file.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Write a haiku about LangGraph and save it to /haiku.md."}]}
)

# tools the agent decided to call
for message in result["messages"]:
    for call in getattr(message, "tool_calls", None) or []:
        print("tool call:", call["name"], call["args"].get("file_path", ""))

print("\nFiles in the agent state:", list(result["files"]))
print(result["files"]["/haiku.md"]["content"])

tool call: write_file /haiku.md

Files in the agent state: ['/haiku.md']
LangGraph weaves bright nodes
Connections spark fluent paths
Meaning flows like light



### 2. Subagents: a researcher and a critic
A subagent is described by a dictionary: `name`, `description` (the main agent uses it to decide when to delegate), `system_prompt` and optional `tools`. The main agent calls a subagent with the built-in `task` tool. The subagent works in its own context and returns only its final answer.

In [4]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(max_results=3)

researcher = {
    "name": "researcher",
    "description": "Searches the web and returns concise, sourced facts on one focused question.",
    "system_prompt": (
        "You are a researcher. Use web search to answer the question you get. "
        "Return 5-8 short bullet points with facts and the source URL for each."
    ),
    "tools": [web_search],
}

critic = {
    "name": "critic",
    "description": "Reviews a draft report saved in a file and lists concrete improvements.",
    "system_prompt": (
        "You are a demanding editor. Read the file you are pointed to and return "
        "at most 5 concrete, actionable improvements. Do not rewrite the report."
    ),
}

analyst = create_deep_agent(
    model="openai:gpt-5-mini",
    subagents=[researcher, critic],
    system_prompt=(
        "You are a lead analyst who writes short research reports.\n"
        "1. Delegate research questions to the `researcher` subagent.\n"
        "2. Write a draft report to /report.md (max 300 words, with a Sources section).\n"
        "3. Ask the `critic` subagent to review /report.md.\n"
        "4. Apply the feedback and save the final version to /report.md.\n"
        "Answer the user with a two-sentence summary."
    ),
)

### 3. Run the agent and watch the delegation
We stream two modes at once: `updates` to print every tool call (subagent delegations and file operations) and `values` to keep the final state.

In [5]:
question = (
    "What are the main differences between LangChain's create_agent and LangGraph's StateGraph, "
    "and when should each be used?"
)

final_state = None
for mode, chunk in analyst.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode=["updates", "values"],
):
    if mode == "values":
        final_state = chunk
        continue
    for node, update in chunk.items():
        messages = (update or {}).get("messages", [])
        messages = getattr(messages, "value", messages)  # some updates wrap the list
        for message in messages:
            for call in getattr(message, "tool_calls", None) or []:
                target = call["args"].get("subagent_type") or call["args"].get("file_path", "")
                print(f"[{node}] {call['name']} -> {target}")

print("\nAgent answer:\n", final_state["messages"][-1].text)

[model] task -> researcher
[model] write_file -> /report.md
[model] task -> critic
[model] read_file -> /report.md
[model] edit_file -> /report.md

Agent answer:
 LangChain's create_agent is a high-level agent loop for building conversational, tool-calling assistants quickly (model + tools + middleware, optimized for developer velocity and per-session/request workflows). LangGraph's StateGraph is a lower-level, Pregel-like orchestration runtime with an explicit typed State, reducer-based updates, and parallel supersteps—use it when you need deterministic, stateful, parallel orchestration or complex pipelines (or combine both: agent front-end + StateGraph backend for heavy orchestration).


### 4. The final report from the virtual file system

In [6]:
from IPython.display import Markdown

Markdown(final_state["files"]["/report.md"]["content"])

Title: When to use LangChain create_agent vs LangGraph StateGraph

Summary
LangChain’s create_agent is a high-level agent harness (part of LangChain’s agents module) for building conversational assistants that call external tools. It provides an agent loop, tool integration, middleware hooks, and sensible defaults to speed prototyping and single-request or multi-turn conversational flows. LangGraph’s StateGraph is a lower-level, graph-based orchestration runtime that adopts bulk-synchronous (Pregel-like) semantics: developers define typed State, node behaviors, and reducer-style state updates to enable parallel supersteps, controlled merges, and explicit orchestration for large or deterministic stateful workflows. Each approach targets different trade-offs: create_agent emphasizes developer velocity and built-in agent features; StateGraph emphasizes explicit state control, parallel execution semantics, and deterministic reduction when reducers are pure.

Differences (high level)
- Abstraction: create_agent ≈ an agent loop abstraction that orchestrates model prompts, tool calls, and middleware callbacks. StateGraph ≈ an explicit graph of nodes that read/write a shared typed State through reducer-style updates and message passing.
- Programming model: create_agent hides orchestration details behind an agent loop and middleware; you implement tools and the agent decides tool invocation. StateGraph requires explicit node definitions, channels/messages, and a compile/execute lifecycle that runs in synchronous supersteps.
- Use cases: create_agent is best for chatbots, tool-enabled assistants, and rapid prototyping where developer velocity and built-in agent behavior matter. StateGraph is best for pipelines, parallel data-orchestration, deterministic multi-step business logic, and graph algorithms that benefit from synchronized supersteps and reducer semantics.
- Concurrency & determinism: create_agent workflows are generally request-driven and often operate sequentially per agent loop, but can use async/parallel tools or external concurrency primitives depending on tool implementations. StateGraph implements bulk-synchronous execution (supersteps) and can produce deterministic outcomes when node updates/reducers are pure and free of non-deterministic side effects; non-pure reducers or external I/O can introduce non-determinism.
- Operational trade-offs: create_agent minimizes orchestration boilerplate and includes middleware and agent-focused tooling; StateGraph demands more upfront design (state schema, reducers, failure/retry policies) but provides clearer execution semantics for large-scale or strongly-typed orchestrations.

Guidance
- When to pick create_agent: choose create_agent for conversational assistants, quick prototypes, or use-cases where built-in agent features (tool integration, middleware hooks, prompt orchestration) reduce development time. Pros: fast iteration, less boilerplate. Watchouts: harder to reason about global state or reproduce exact execution if many side effects exist.
- When to pick StateGraph: choose StateGraph for large-scale or latency-tolerant pipelines, deterministic workflows that require explicit state merges, parallel graph algorithms, or when you need strong control over state schema, merge/reducer logic, and superstep execution. Pros: clearer execution semantics, easier reasoning about parallel updates. Watchouts: more upfront design, testing reducers for purity, and handling external I/O in a way that preserves determinism.
- Practical tips & migration notes: map agent state fields (e.g., conversation or session variables) to explicit StateGraph keys; implement each tool or step as a node (or node + side-effecting worker) and encode state transitions as reducers. Add comprehensive unit tests for reducers to ensure deterministic behavior. Instrument observability (per-superstep logs, state diffs) and define failure/retry strategies for nodes that perform I/O. Consider a hybrid approach: keep the agent loop for conversational front-end and implement complex orchestration or heavy parallel processing using StateGraph for backend workflows.

Sources
- https://docs.langchain.com/oss/python/langchain/overview
- https://docs.langchain.com/oss/python/langchain/middleware/built-in
- https://langchain-ai.github.io/langgraphjs/reference/classes/langgraph.StateGraph.html
- https://docs.langchain.com/oss/python/langgraph/pregel
- https://www.langchain.com/blog/langgraph


### When to use what?
| Need | Tool |
|---|---|
| Model + tools in a loop, short tasks | `langchain.agents.create_agent` |
| Long, multi-step tasks with notes, files and delegation | `deepagents.create_deep_agent` |
| Fully custom control flow (fixed steps, loops, branching) | LangGraph `StateGraph` |

Next steps: give the agent long-term memory (`memory=`), skills (`skills=`), human approval for selected tools (`interrupt_on=`) or a real file system / sandbox through a different `backend`.